# 第3章 NumPy数组计算 · 课堂代码

> 本 notebook 与课件《Python金融数据分析 · 第3章 NumPy数组计算》配套。
> 内容改编自《Python金融大数据分析（第2版）》第4章。

**使用说明**
- 点击单元格，按 `Shift + Enter` 运行；
- `%timeit` 是 Notebook 的计时"魔法命令"，只能在 Notebook 中运行。

新增部分为自编教学例子，参考 Wes McKinney《Python for Data Analysis》第3版的主题组织：[在线书](https://wesmckinney.com/book/)。先预测输出，再执行；练习答案可展开查看。

## 1. 为什么需要 NumPy？

上一章我们用循环计算日收益率：只有 4 个价格没问题，但真实数据动辄数千个交易日、成百上千只股票——循环又慢又啰嗦。NumPy 的目标：**把 4 行循环压缩成 1 行，并且快上百倍**。

In [1]:
# 上一章的做法：for 循环
prices = [100.0, 102.0, 101.0, 105.0]
returns = []
for i in range(1, len(prices)):
    r = (prices[i] - prices[i-1]) / prices[i-1]
    returns.append(r)
returns

[0.02, -0.00980392156862745, 0.039603960396039604]

NumPy（Numerical Python）是 Python 科学计算的基石。核心对象 `ndarray` 底层用 C 实现，运算直接作用于整块内存。

`import numpy as np` 是全世界通用的写法，请照抄。

In [2]:
import numpy as np     # 约定俗成的导入方式

prices = np.array([100.0, 102.0, 101.0, 105.0])
returns = prices[1:] / prices[:-1] - 1    # 一行搞定！
returns

array([ 0.02      , -0.00980392,  0.03960396])

## 2. 创建数组

### 2.1 从列表创建与基本属性

- `np.array(列表)`：把 Python 列表转换为数组；
- **类型统一**：数组中所有元素同一类型，这正是快的原因；
- `shape` 描述"形状"，是理解二维数据的关键。

In [3]:
a = np.array([100.0, 102.0, 101.0, 105.0])
print(a)           # array([100., 102., 101., 105.])

print(type(a))     # numpy.ndarray
print(a.shape)     # (4,)      形状：长度为4的一维数组
print(a.dtype)     # float64   元素类型：全数组统一
print(len(a))      # 4

[100. 102. 101. 105.]
<class 'numpy.ndarray'>
(4,)
float64
4


### 2.2 快速生成常用数组

`linspace(起, 止, 个数)` 在画利率区间、画函数图像时常用。例如给一组折现率，批量观察债券价格变化。

In [4]:
print(np.zeros(5))           # array([0., 0., 0., 0., 0.])
print(np.ones(3))            # array([1., 1., 1.])

print(np.arange(0, 10, 2))   # array([0, 2, 4, 6, 8])  类似range
print(np.linspace(0, 1, 5))  # 0到1之间均匀取5个点（含端点）

[0. 0. 0. 0. 0.]
[1. 1. 1.]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


### 2.3 二维数组：数据的"表格"形态

列表中套列表 → 二维数组，可以看作**表格/矩阵**。金融数据的典型形态：**行=日期，列=资产**。

In [5]:
# 3天 x 2只股票 的收盘价：每行一天，每列一只股票
p = np.array([[25.0, 100.0],
              [25.5,  99.0],
              [26.0, 102.0]])

print(p.shape)     # (3, 2)   3行2列
print(p.ndim)      # 2        维度数
print(p.size)      # 6        元素总数

(3, 2)
2
6


### 2.3A 用成绩表理解 shape 与 axis

每行是一名学生，每列是一门课。axis=0 沿学生方向汇总，得到各科均分；axis=1 沿科目方向汇总，得到各学生均分。

In [6]:
scores = np.array([[80, 90, 70], [60, 75, 90]])
print(scores.shape)        # 2名学生、3门课
print(scores.mean(axis=0)) # [70. 82.5 80.]
print(scores.mean(axis=1)) # [80. 75.]

(2, 3)
[70.  82.5 80. ]
[80. 75.]


### 2.4 形状变换与拼接

**易错点**：数组的 `+` 是**逐元素相加**，不是拼接！拼接请用 `np.concatenate`。

In [7]:
a = np.arange(6)             # [0 1 2 3 4 5]
print(a.reshape(2, 3))       # 变成2行3列
print(a.reshape(-1, 2))      # -1 表示"自动算"：3行2列

print(np.array([1, 2]) + np.array([3, 4]))   # [4 6]，不是拼接！
print(np.concatenate([np.array([1, 2]), np.array([3, 4])]))  # [1 2 3 4]

[[0 1 2]
 [3 4 5]]
[[0 1]
 [2 3]
 [4 5]]
[4 6]
[1 2 3 4]


## 3. 索引与切片

### 3.1 一维数组：与列表几乎相同

`prices[1:]` 与 `prices[:-1]` 这对组合后面算收益率要用，务必熟练。

注意：数组切片返回的是**视图**（与原数组共享内存），修改切片会影响原数组，与列表不同。

In [8]:
prices = np.array([30.1, 30.5, 29.8, 31.2, 30.9])

print(prices[0])       # 30.1    索引从0开始
print(prices[-1])      # 30.9    倒数第一个
print(prices[1:3])     # array([30.5, 29.8])   含头不含尾
print(prices[1:])      # 除第一个外的所有元素
print(prices[:-1])     # 除最后一个外的所有元素

30.1
30.9
[30.5 29.8]
[30.5 29.8 31.2 30.9]
[30.1 30.5 29.8 31.2]


### 3.2 二维数组：先行后列

写法：`数组[行, 列]`；冒号 `:` 表示"全部"。`p[:, 1]` 即"取出某只股票的整个价格序列"，是金融分析中的高频操作。

In [9]:
p = np.array([[25.0, 100.0],
              [25.5,  99.0],
              [26.0, 102.0]])

print(p[0, 0])     # 25.0   第0行第0列
print(p[1, 1])     # 99.0   第1行第1列
print(p[0])        # 第0行 -> array([25., 100.])
print(p[:, 1])     # 所有行、第1列 -> 股票2的价格序列
print(p[1:, :])    # 第1行到末尾的所有行

25.0
99.0
[ 25. 100.]
[100.  99. 102.]
[[ 25.5  99. ]
 [ 26.  102. ]]


### 3.3 切片视图与显式复制

基本切片通常共享原数组的数据。需要独立修改时加 copy()。这与 Python 列表的切片不同。

In [10]:
measurements = np.array([10, 20, 30, 40])
view = measurements[1:3]
independent = measurements[1:3].copy()
view[0] = 99
print(measurements)  # [10 99 30 40]
print(independent)   # [20 30]

[10 99 30 40]
[20 30]


**先动手**：改成 measurements[[1, 2]] 后再修改结果，原数组会变化吗？

In [11]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
selected = measurements[[1, 2]]
selected[0] = -1
print(measurements)  # 不变；整数数组索引产生复制。
```
</details>

## 4. 向量化运算

### 4.1 数组与标量：整体运算

**向量化**：运算自动作用于数组中的**每一个**元素，无需循环；结果是新数组，原数组不变。可以把数组想象成"一整列Excel数据同时参与计算"。

In [12]:
prices = np.array([100.0, 102.0, 101.0, 105.0])

print(prices + 1)       # 每个元素加1
print(prices * 1.05)    # 每个元素涨5%
print(prices ** 2)      # 每个元素平方
print(1 / prices)       # 每个元素取倒数

[101. 103. 102. 106.]
[105.   107.1  106.05 110.25]
[10000. 10404. 10201. 11025.]
[0.01       0.00980392 0.00990099 0.00952381]


### 4.2 数组与数组：逐元素配对

两个数组运算：**位置一一对应**。参与运算的两个数组形状必须一致（或满足广播规则），否则报错。

In [13]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([10., 20., 30.])

print(a + b)       # array([11., 22., 33.])
print(a * b)       # array([10., 40., 90.])
print(b / a)       # array([10., 10., 10.])

[11. 22. 33.]
[10. 40. 90.]
[10. 10. 10.]


### 4.2A 广播：每科加不同的分数

(2, 3) 与 (3,) 可以逐列配对。广播从末尾维度比较，长度相同或其中一个为 1 才能兼容。

In [14]:
scores = np.array([[80, 90, 70], [60, 75, 90]])
bonus = np.array([2, 0, 5])
adjusted = scores + bonus
print(adjusted)
student_mean = scores.mean(axis=1, keepdims=True)
print(student_mean.shape)  # (2, 1)
print(scores - student_mean)

[[82 90 75]
 [62 75 95]]
(2, 1)
[[  0.  10. -10.]
 [-15.   0.  15.]]


**先动手**：将每一科减去该科均分；解释结果为什么每列均值为 0。

In [15]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
centered = scores - scores.mean(axis=0)
print(centered)
print(centered.mean(axis=0))  # [0. 0. 0.]
```
</details>

### 4.3 例：一行代码计算收益率

`prices[1:]` 是今日价序列，`prices[:-1]` 是昨日价序列，相除即逐日配对。上一章的 4 行循环，现在 **1 行**。

In [16]:
prices = np.array([100.0, 102.0, 101.0, 105.0])

# 简单收益率：(今日-昨日)/昨日
ret = prices[1:] / prices[:-1] - 1
print(ret)        # array([0.02, -0.0098, 0.0396])

# 对数收益率（后文常用）
logret = np.log(prices[1:] / prices[:-1])
print(logret)

[ 0.02       -0.00980392  0.03960396]
[ 0.01980263 -0.0098523   0.03883983]


### 4.4 快多少？速度对比

`%timeit` 会自动多次运行取平均，是 Notebook 的计时魔法命令。

In [17]:
big = np.arange(1_000_000)          # 0到999999

%timeit [x ** 2 for x in big]       # Python循环：约几百毫秒
%timeit big ** 2                    # NumPy向量化：约1毫秒

45.4 ms ± 519 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


876 μs ± 21.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


**经验法则**：看到"对每个元素做同样的事"，先想向量化，再考虑循环。

### 4.5 常用向量化数学函数

| 函数 | 作用 | 金融用途举例 |
| --- | --- | --- |
| `np.log(x)` / `np.exp(x)` | 对数 / 指数 | 对数收益率、连续复利 |
| `np.sqrt(x)` | 平方根 | 波动率换算 |
| `np.abs(x)` | 绝对值 | 偏差幅度 |
| `np.maximum(a, b)` | 逐元素取大 | 期权收益 max(S-K, 0) |
| `np.round(x, n)` | 四舍五入 | 报表展示 |
| `np.cumsum(x)` | 累计和 | 累计收益、净值曲线 |
| `np.diff(x)` | 相邻差分 | 价格变动额 |

In [18]:
# 期权到期收益：行权价 K，到期价序列 S
S = np.array([95., 100., 105., 110.])
K = 100
payoff = np.maximum(S - K, 0)
payoff

array([ 0.,  0.,  5., 10.])

## 5. 布尔索引：按条件筛选

### 5.1 比较运算返回布尔数组

数组的比较运算也是**向量化**的。组合条件要用 `&`（且）、`|`（或），且每个条件加括号——数组条件中 `and`、`or` 会报错。

In [19]:
ret = np.array([0.02, -0.01, 0.04, -0.03, 0.01])

ret > 0      # array([True, False, True, False, True])

array([ True, False,  True, False,  True])

In [20]:
(ret > -0.01) & (ret < 0.01)     # 小幅波动日

array([False, False, False, False, False])

### 5.2 用布尔数组筛选与统计

- `数组[布尔数组]`：只保留对应位置为 `True` 的元素；
- `np.sum(布尔数组)`：统计满足条件的个数——非常实用的技巧；
- 金融应用：统计涨停天数、回撤超过阈值的次数、极端波动日。

In [21]:
ret = np.array([0.02, -0.01, 0.04, -0.03, 0.01])

print(ret[ret > 0])          # array([0.02, 0.04, 0.01]) 只留上涨日
print(ret[ret < -0.02])      # array([-0.03])            大跌日

print(np.sum(ret > 0))       # 3     True按1计数：上涨天数
print((ret > 0).mean())      # 0.6   上涨日占比

[0.02 0.04 0.01]
[-0.03]
3
0.6


### 5.3 np.where：条件赋值

`np.where(条件, 真时的值, 假时的值)` 是向量化的 `if/else`。应用：构造交易信号——收益为正记1（持有多头），为负记0。

In [22]:
ret = np.array([0.02, -0.01, 0.04])

print(np.where(ret > 0, 1, 0))        # array([1, 0, 1])
print(np.where(ret > 0, "涨", "跌"))   # array(['涨', '跌', '涨'])

[1 0 1]
['涨' '跌' '涨']


## 6. 统计函数

### 6.1 常用聚合函数

"把一串数压缩成一个数"的函数叫**聚合函数**。**标准差**是金融中风险（波动率）的度量，务必熟悉。

In [23]:
ret = np.array([0.02, -0.01, 0.04, -0.03, 0.01])

print(ret.sum())        # 0.03     求和
print(ret.mean())       # 0.006    均值（日均收益率）
print(ret.std())        # 0.026... 标准差（波动性）
print(ret.max())        # 0.04     最大值
print(ret.min())        # -0.03    最小值
print(ret.argmax())     # 2        最大值的"位置"

0.030000000000000006
0.006000000000000001
0.024166091947189144
0.04
-0.03
2


### 6.2 二维数组与 axis 参数

记忆口诀：**axis 是被"压掉"的那一维**。`axis=0` 结果对应每一**列**；`axis=1` 结果对应每一**行**。

In [24]:
# 3天 x 2只股票的日收益率
r = np.array([[ 0.02, -0.01],
              [-0.01,  0.03],
              [ 0.04,  0.01]])

print(r.mean())          # 0.0133...  全部元素的均值
print(r.mean(axis=0))    # 按"列"求均值 -> 每只股票的日均收益
print(r.mean(axis=1))    # 按"行"求均值 -> 每天的组合平均收益

0.013333333333333334
[0.01666667 0.01      ]
[0.005 0.01  0.025]


### 6.3 累计与差分

- `np.diff`：相邻元素相减，长度少1——快速得到每日变动；
- `np.cumsum`：逐步累加——从"每日现金流"得到"累计现金流"。

In [25]:
prices = np.array([100., 102., 101., 105.])

print(np.diff(prices))          # array([2., -1., 4.]) 每日涨跌"额"
print(np.cumsum([1, 1, 1, 1]))  # array([1, 2, 3, 4]) 累计和

[ 2. -1.  4.]
[1 2 3 4]


### 6.4 非金融小案例：左右走动

每步向左或向右移动一格，起点为 0。先生成步长，再用 cumsum 得到位置；随机种子使课堂输出可复现。

In [26]:
walk_rng = np.random.default_rng(7)
directions = walk_rng.integers(0, 2, size=12)
steps = np.where(directions == 0, -1, 1)
positions = np.r_[0, steps.cumsum()]
print(steps)
print(positions)
print("最远距离：", np.abs(positions).max())

[ 1  1  1  1  1  1  1 -1 -1 -1 -1  1]
[0 1 2 3 4 5 6 7 6 5 4 3 4]
最远距离： 7


**先动手**：改成 100 步，并计算走动后位于起点右侧的比例；分母是否包括起点？

In [27]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
steps100 = np.where(walk_rng.integers(0, 2, 100) == 0, -1, 1)
positions100 = steps100.cumsum()
print((positions100 > 0).mean())  # 分母100，不含起点
```
</details>

## 7. 综合案例：模拟股价并测风险

**年化波动率 = 日标准差 × √252**，业界标准做法。

In [28]:
rng = np.random.default_rng(8)          # 随机数生成器(固定种子)
n = 250                                  # 250个交易日

# 模拟日收益率：均值0.04%，日波动1.5%
rets = 0.0004 + 0.015 * rng.standard_normal(n)

# 由收益率还原价格曲线（期初100元）
prices = 100 * np.exp(np.cumsum(rets))

print(f"日均收益率：{rets.mean():.4%}")
print(f"日波动率：  {rets.std():.4%}")
print(f"年化波动率：{rets.std()*np.sqrt(252):.2%}")
print(f"最高价/最低价：{prices.max():.2f} / {prices.min():.2f}")

日均收益率：0.0492%
日波动率：  1.5982%
年化波动率：25.37%
最高价/最低价：117.50 / 85.64


进一步分析：上涨日占比、极端下跌日、最大回撤。

`np.maximum.accumulate` 给出去到每个时点为止的历史最高价；**最大回撤**是基金与策略最常用的风险指标之一。

In [29]:
# 上涨日占比
print(f"上涨日占比：{(rets > 0).mean():.2%}")

# 单日跌幅超过3%的"极端日"
extreme = rets[rets < -0.03]
print(f"极端下跌天数：{len(extreme)}")

# 最大回撤的粗略观察：历史最高价 vs 当前价
peak = np.maximum.accumulate(prices)
drawdown = prices / peak - 1
print(f"最大回撤：{drawdown.min():.2%}")

上涨日占比：50.00%
极端下跌天数：8
最大回撤：-14.24%


## 8. 课后任务

1. 运行本 notebook，修改参数观察输出变化；
2. 给定价格数组 `np.array([50, 52, 51, 55, 53, 56])`：计算日收益率、找出最大单日涨幅、统计上涨天数；
3. 选做：用 `np.where` 构造信号"收益为正买1手、为负卖1手"，并计算该策略的每日收益（信号×次日收益，注意错位）。